In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

housebuilding = spark.table("scottish_housing_ws.silver.housebuilding")

In [0]:
print(f"Row count: {housebuilding.count()}")
housebuilding.show(3)

In [0]:
starts = (
    housebuilding
    .filter(F.col("build_type") == "Starts")
    .select(
        "report_date",
        "year_quarter",
        "area_code",
        "area_name",
        "area_type",
        "housing_sector",
        F.col("dwellings").alias("starts")
    )
)

completions = (
    housebuilding
    .filter(F.col("build_type") == "Completions")
    .select(
        "year_quarter",
        "area_code",
        "housing_sector",
        F.col("dwellings").alias("completions")
    )
)

approvals = (
    housebuilding
    .filter(F.col("build_type") == "Approvals")
    .select(
        "year_quarter",
        "area_code",
        "housing_sector",
        F.col("dwellings").alias("approvals")
    )
)

In [0]:
gold = (
    starts
    .join(completions, on=["year_quarter", "area_code", "housing_sector"], how="left")
    .join(approvals, on=["year_quarter", "area_code", "housing_sector"], how="left")
    .withColumn(
        "starts_completions_ratio",
        F.round(
            F.when(F.col("completions") == 0, None)
            .otherwise(F.col("starts") / F.col("completions")),
            2
        )
    )
    .withColumn(
        "starts_lag_4q",
        F.lag("starts", 4).over(
            Window
            .partitionBy("area_code", "housing_sector")
            .orderBy("year_quarter")
        )
    )
    .select(
        "report_date",
        "year_quarter",
        "area_code",
        "area_name",
        "area_type",
        "housing_sector",
        "starts",
        "completions",
        "approvals",
        "starts_completions_ratio",
        "starts_lag_4q"
    )
    .orderBy("area_code", "housing_sector", "year_quarter")
)

In [0]:
print(f"Row count: {gold.count()}")

# Scotland national, all sectors
gold.filter(
    (F.col("area_type") == "Country") &
    (F.col("housing_sector") == "All")
).orderBy("year_quarter").show(5)

gold.filter(
    (F.col("area_type") == "Country") &
    (F.col("housing_sector") == "All")
).orderBy(F.col("year_quarter").desc()).show(5)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.housebuilding_starts_vs_completions_lag")
)

In [0]:
(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_housebuilding_starts_vs_completions_lag/")
)

print("Gold 09 complete.")

In [0]:
# Filter to Scotland national and All sectors 
# Pivot starts vs completions onto the same row
# Focus on All sectors + Country level for the headline numbers
# Keep council area + sector breakdown for drill-down